In [ ]:
from collections import Counter

# Q2.2.1 Training corpus given in the assignment
corpus = (
    "low low low low low lowest lowest "
    "newer newer newer newer newer newer "
    "wider wider wider new new"
)

# Count how many times each word appears in the corpus.
word_counts = Counter(corpus.split())

print("Word frequencies:")
print(word_counts)

# Preferred order is used only when two pairs have the same frequency.
preferred_pairs = [
    ("e", "r"),
    ("er", "_"),
    ("n", "e"),
    ("ne", "w"),
    ("l", "o"),
    ("lo", "w"),
    ("new", "er_"),
    ("low", "_"),
    ("w", "i"),
    ("wi", "d")
]


def create_tokens(word):
    # Break the word into characters and add the end marker.
    return list(word) + ["_"]


def find_pair_counts(token_data, frequencies):
    # Count adjacent token pairs using their word frequencies.
    counts = Counter()

    for word, frequency in frequencies.items():
        pieces = token_data[word]

        for first, second in zip(pieces, pieces[1:]):
            counts[(first, second)] += frequency

    return counts


def select_pair(pair_counts):
    # Find the pair with the highest frequency.
    highest = max(pair_counts.values())

    candidates = {
        pair
        for pair, count in pair_counts.items()
        if count == highest
    }

    # Use the predefined order to handle ties.
    for pair in preferred_pairs:
        if pair in candidates:
            return pair, highest

    return sorted(candidates)[0], highest


def apply_merge(token_data, selected_pair):
    # Replace the selected pair with one combined token.
    updated_data = {}

    for word, pieces in token_data.items():
        result = []
        position = 0

        while position < len(pieces):
            if (
                position + 1 < len(pieces)
                and (pieces[position], pieces[position + 1])
                == selected_pair
            ):
                result.append(
                    pieces[position] + pieces[position + 1]
                )
                position += 2
            else:
                result.append(pieces[position])
                position += 1

        updated_data[word] = result

    return updated_data


def current_vocabulary(token_data):
    # Collect all tokens that currently exist.
    vocabulary = set()

    for pieces in token_data.values():
        vocabulary.update(pieces)

    return sorted(vocabulary)


def train_mini_bpe(frequencies, merge_count):
    # Start with character-level tokens.
    token_data = {
        word: create_tokens(word)
        for word in frequencies
    }

    merge_list = []

    for step in range(1, merge_count + 1):
        pair_counts = find_pair_counts(
            token_data,
            frequencies
        )

        if not pair_counts:
            break

        selected, frequency = select_pair(pair_counts)
        merge_list.append(selected)

        token_data = apply_merge(
            token_data,
            selected
        )

        vocab_size = len(
            current_vocabulary(token_data)
        )

        print(
            f"Step {step}: "
            f"top pair = {selected}, "
            f"count = {frequency}, "
            f"vocabulary size = {vocab_size}"
        )

    return merge_list


def split_new_word(word, merge_list):
    # Apply the learned merges to a new word.
    pieces = create_tokens(word)

    for selected_pair in merge_list:
        result = []
        position = 0

        while position < len(pieces):
            if (
                position + 1 < len(pieces)
                and (pieces[position], pieces[position + 1])
                == selected_pair
            ):
                result.append(
                    pieces[position] + pieces[position + 1]
                )
                position += 2
            else:
                result.append(pieces[position])
                position += 1

        pieces = result

    return pieces


if __name__ == "__main__":
    print("Q2.2 Mini-BPE Learning")
    print("======================")

    learned_pairs = train_mini_bpe(
        training_words,
        10
    )

    print("\nQ2.2.2 Word Segmentation")
    print("========================")

    test_words = [
        "new",
        "newer",
        "lowest",
        "widest",
        "newestest"
    ]

    for word in test_words:
        print(
            f"{word}: "
            f"{split_new_word(word, learned_pairs)}"
        )

Word Frequencies
----------------
Counter({'newer': 6, 'low': 5, 'wider': 3, 'lowest': 2, 'new': 2})

Q2.2 Mini-BPE Learning
Step 1: top pair = ('e', 'r'), count = 9, vocabulary size = 11
Step 2: top pair = ('er', '_'), count = 9, vocabulary size = 11
Step 3: top pair = ('n', 'e'), count = 8, vocabulary size = 11
Step 4: top pair = ('ne', 'w'), count = 8, vocabulary size = 11
Step 5: top pair = ('l', 'o'), count = 7, vocabulary size = 10
Step 6: top pair = ('lo', 'w'), count = 7, vocabulary size = 10
Step 7: top pair = ('new', 'er_'), count = 6, vocabulary size = 11
Step 8: top pair = ('low', '_'), count = 5, vocabulary size = 12
Step 9: top pair = ('w', 'i'), count = 3, vocabulary size = 11
Step 10: top pair = ('wi', 'd'), count = 3, vocabulary size = 10

Q2.2.2 Word Segmentation
new: ['new', '_']
newer: ['newer_']
lowest: ['low', 'e', 's', 't', '_']
widest: ['wid', 'e', 's', 't', '_']
newestest: ['new', 'e', 's', 't', 'e', 's', 't', '_']
